# Melanoma experiments

Right panel: **Accelerator = GPU T4 x2**, **Internet = On**.

Do NOT pick P100. Kaggle's PyTorch dropped Pascal support, so a P100 reports
`cuda True` and then refuses to run anything.

Resumable: each finished fold is written to results.csv, so if the session dies
just Run All again and it continues.


In [ ]:
import os, shutil, time
t0 = time.time()

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Right panel > Accelerator > GPU T4 x2.")

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
supported = torch.cuda.get_arch_list()
print("gpu:", name, f"({arch})")

# torch.cuda.is_available() is NOT enough. A P100 is sm_60, which current
# Kaggle PyTorch builds no longer support: is_available() returns True and then
# every kernel launch fails. Check the architecture is actually compiled in.
if arch not in supported:
    raise SystemExit(
        f"{name} is {arch}, which this PyTorch does not support.\n"
        f"Supported: {supported}\n"
        "Switch the accelerator to GPU T4 x2.")

try:
    import albumentations as A
    A.Affine
    print("albumentations", A.__version__)
except Exception:
    import subprocess
    subprocess.run(["pip", "install", "-q", "albumentations>=2.0"], check=False)
    import albumentations as A
    print("albumentations installed:", A.__version__)


In [ ]:
# --- find the data, whatever depth Kaggle mounts it at ------------------
# Kaggle has mounted datasets at /kaggle/input/<slug>/ and at
# /kaggle/input/datasets/<user>/<slug>/, and it auto-extracts .tar uploads,
# which adds another level. So walk instead of assuming a depth.
#
# The pruning line matters: train_512 holds 57,855 files on a network mount,
# and descending into it turns a 2 second search into several minutes.
FOLDS = None
IMG_DIR = None
py_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    if IMG_DIR is None and "train_512" in dirs:
        IMG_DIR = os.path.join(root, "train_512")
    if FOLDS is None and "folds.csv" in files:
        FOLDS = os.path.join(root, "folds.csv")
    py_files += [os.path.join(root, f) for f in files if f.endswith(".py")]
    dirs[:] = [d for d in dirs if d not in ("train_512", "test_512")]

assert FOLDS, "folds.csv not found under /kaggle/input. Add the dataset, then restart the session."
assert IMG_DIR, "train_512 not found under /kaggle/input"

WORK = "/kaggle/working"
SRC = os.path.join(WORK, "src")
os.makedirs(SRC, exist_ok=True)
for p in py_files:
    shutil.copy(p, SRC)

print("img_dir :", IMG_DIR)
print("folds   :", FOLDS)
print("copied  :", len(py_files), "source files")
assert len(py_files) >= 5, "source .py files missing from the dataset"


In [ ]:
# --- run -----------------------------------------------------------------
# Both experiments over the same folds, rather than all folds of one. The
# runner loops experiments outer and folds inner, so if it runs out of time it
# would finish the baseline and none of the ablation, losing the comparison.
# Fewer paired folds beats more unpaired ones.
BUDGET = 3.6

!cd {SRC} && python experiment_runner.py \
    --img_dir "{IMG_DIR}" \
    --folds_csv "{FOLDS}" \
    --out_dir {WORK}/reports \
    --only resnet34_224_ext,resnet34_224_noext \
    --folds 0,1,2 \
    --batch_size 64 \
    --num_workers 4 \
    --time_budget_h {BUDGET}


In [ ]:
!cd {SRC} && python report_results.py --in_dir {WORK}/reports

import pandas as pd
pd.read_csv(f"{WORK}/reports/results.csv")
